In [1]:
8+8

16

In [2]:
import pandas as pd 
import tensorflow as tf
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [3]:
data=pd.read_csv('huge_1M_titanic.csv')

In [4]:
data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [5]:
data=data.drop(columns=['PassengerId','Name','Age','Ticket','Cabin'])

In [6]:
data

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
0,1,1,female,0,0,76.760165,C
1,0,3,male,0,0,10.193097,S
2,0,3,male,0,0,12.029416,C
3,0,3,male,0,0,13.429448,S
4,0,3,male,0,0,4.840769,C
...,...,...,...,...,...,...,...
999995,0,2,male,0,0,26.196512,S
999996,1,1,female,0,0,102.555248,S
999997,0,3,male,1,0,8.274807,S
999998,0,2,male,1,0,0.000000,S


In [7]:
data['Embarked'].mode()

0    S
Name: Embarked, dtype: str

In [8]:
data['Embarked']=data['Embarked'].replace({'S':'Southamton','C':'Cheboung','Q':'Queensland'})

In [9]:
data.dropna(subset=['Embarked'], inplace=True)

In [14]:
label=LabelEncoder() 
onehot=OneHotEncoder(sparse_output=False)
scale=StandardScaler()

In [13]:
data.sample()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked
366484,1,1,0,1,0,104.646726,Cheboung


In [12]:
data['Sex']=label.fit_transform(data['Sex'])

In [15]:
Embarked=onehot.fit_transform(data[['Embarked']])


In [16]:
Embarked=pd.DataFrame(Embarked,columns=onehot.get_feature_names_out())

In [17]:
data=pd.concat([data.drop(columns=['Embarked']), Embarked], axis=1)

In [18]:
data.sample()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cheboung,Embarked_Queensland,Embarked_Southamton
76741,0.0,2.0,1.0,0.0,0.0,3.19395,0.0,0.0,1.0


In [21]:
data.isnull().sum()

Survived               0
Pclass                 0
Sex                    0
SibSp                  0
Parch                  0
Fare                   0
Embarked_Cheboung      0
Embarked_Queensland    0
Embarked_Southamton    0
dtype: int64

In [20]:
data=data.dropna()

In [22]:
num_cols=['Pclass','SibSp','Parch','Fare']

In [23]:
data[num_cols]=scale.fit_transform(data[num_cols])

In [24]:
import pickle

with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label,file)
with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot,file)
with open('scale.pkl','wb') as file:
    pickle.dump(scale,file)

In [25]:
data

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cheboung,Embarked_Queensland,Embarked_Southamton
0,1.0,-1.568955,0.0,-0.462632,-0.469293,0.902298,1.0,0.0,0.0
1,0.0,0.824071,1.0,-0.462632,-0.469293,-0.485006,0.0,0.0,1.0
2,0.0,0.824071,1.0,-0.462632,-0.469293,-0.446736,1.0,0.0,0.0
3,0.0,0.824071,1.0,-0.462632,-0.469293,-0.417558,0.0,0.0,1.0
4,0.0,0.824071,1.0,-0.462632,-0.469293,-0.596552,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
997755,0.0,0.824071,0.0,0.442326,0.833900,-0.399056,0.0,0.0,1.0
997756,1.0,0.824071,1.0,0.442326,0.833900,-0.435926,0.0,0.0,1.0
997757,0.0,0.824071,0.0,6.777027,2.137092,0.760470,0.0,0.0,1.0
997758,0.0,0.824071,1.0,-0.462632,-0.469293,-0.592560,0.0,0.0,1.0


In [26]:
X=data.drop(columns='Survived')
Y=data['Survived']

In [27]:
X.sample()

,Pclass,Sex,SibSp,Parch,Fare,Embarked_Cheboung,Embarked_Queensland,Embarked_Southamton
888074,0.824071,1.0,-0.462632,-0.469293,-0.434625,1.0,0.0,0.0


In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=20,random_state=42)

In [29]:
X_train,X_valid,Y_train,Y_valid=train_test_split(X_train,Y_train,test_size=0.2,random_state=42)

In [30]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [31]:
(X_train.shape[1],)

(8,)

In [32]:
model = Sequential([Dense(128,input_shape=(X_train.shape[1],),activation='relu'),
                    Dense(64,activation='relu'),
                    Dense(32,activation='relu'),
                    Dense(1,activation='sigmoid')])

In [34]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,521 (45.00 KB)

 Trainable params: 11,521 (45.00 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

In [38]:
model.fit(X_train,Y_train, validation_data=(X_valid,Y_valid), epochs=3)

Epoch 1/3
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 117s 5ms/step - accuracy: 0.8483 - loss: 0.3424 - val_accuracy: 0.8562 - val_loss: 0.3264
Epoch 2/3
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 118s 5ms/step - accuracy: 0.8554 - loss: 0.3275 - val_accuracy: 0.8578 - val_loss: 0.3226
Epoch 3/3
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 134s 5ms/step - accuracy: 0.8563 - loss: 0.3251 - val_accuracy: 0.8582 - val_loss: 0.3222


In [40]:
def build_model(hp):
    model = Sequential([Dense(64,input_shape=(X_train.shape[1],),activation='relu'),
                        Dense(32,activation='relu'),
                        Dense(1,activation='sigmoid')])
    optimizer=hp.Choice('optimizer',values=['Adam','sgd','RMSprop','AdamW','Adamax','Adagrad'])
    model.compile(optimizer=optimizer,loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [41]:
tuner=kt.RandomSearch(build_model,max_trials=5,objective='val_accuracy')

In [42]:
tuner.search(X_train,Y_train,validation_data=(X_test,Y_test), epochs=3)


Trial 2 Complete [00h 00m 00s]

Best val_accuracy So Far: None
Total elapsed time: 00h 00m 00s

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
Adagrad           |AdamW             |optimizer



Traceback (most recent call last):
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hparams_api
ModuleNotFoundError: No module named 'tensorboard'


RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
  File "c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hparams_api
ModuleNotFo

In [43]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'AdamW'}

In [44]:
model=tuner.get_best_models(num_models=1)[0]

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'untitled_project\trial_0\checkpoint.weights.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [45]:
model.fit(X_train,Y_train,validation_data=(X_test,Y_test), epochs=10, initial_epoch=3)

Epoch 4/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 107s 4ms/step - accuracy: 0.8565 - loss: 0.3240 - val_accuracy: 0.8000 - val_loss: 0.5068
Epoch 5/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 152s 6ms/step - accuracy: 0.8567 - loss: 0.3233 - val_accuracy: 0.7500 - val_loss: 0.4972
Epoch 6/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 103s 4ms/step - accuracy: 0.8571 - loss: 0.3228 - val_accuracy: 0.8000 - val_loss: 0.4976
Epoch 7/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 98s 4ms/step - accuracy: 0.8571 - loss: 0.3225 - val_accuracy: 0.7500 - val_loss: 0.5001
Epoch 8/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 89s 4ms/step - accuracy: 0.8570 - loss: 0.3223 - val_accuracy: 0.8000 - val_loss: 0.4767
Epoch 9/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 140s 6ms/step - accuracy: 0.8572 - loss: 0.3220 - val_accuracy: 0.7500 - val_loss: 0.4862
Epoch 10/10
24888/24888 ━━━━━━━━━━━━━━━━━━━━ 86s 3ms/step - accuracy: 0.8573 - loss: 0.3218 - val_accuracy: 0.7500 - val_loss: 0.4900
